# 미분과 기울기 (Differentiation and Gradient)

- 모델은 손실을 줄이려면 무엇을 알아야 할까?
- 가중치와 편향을 어느 방향으로 바꿔야 손실이 줄어들까?

이때 필요한 개념이 바로 미분과 기울기(gradient)이다.

## 1. 평균 변화율

함수의 변화율을 가장 직관적으로 이해하는 방법은
두 점 사이에서 얼마나 변했는지를 보는 것이다.

예를 들어 함수 $f(x)$에서
$x$가 조금 바뀌었을 때
$f(x)$가 얼마나 달라졌는지를 보면
함수의 기울기를 대략적으로 짐작할 수 있다.

평균 변화율은 아래처럼 쓸 수 있다.

$$
\frac{f(x+h) - f(x)}{h}
$$

이 값은 두 점 사이에서의 평균적인 기울기를 의미한다.

In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt

In [2]:
def f(x):
    return x ** 2

x1 = 2
x2 = 3

avg_rate = (f(x2) - f(x1)) / (x2 - x1)
print("평균 변화율 :", avg_rate)

평균 변화율 : 5.0


## 2. 순간 변화율과 미분

평균 변화율은 두 점 사이의 기울기이다.

그런데 우리가 정말 알고 싶은 것은
특정 한 점에서의 순간적인 기울기이다.

이를 위해 $h$를 아주 작게 만들면,
평균 변화율은 특정 점에서의 기울기에 가까워진다.

즉 미분은
입력이 아주 조금 변할 때
출력이 얼마나 변하는지를 보는 도구라고 이해할 수 있다.


In [3]:
def derivative_f(x):
    # f(x) = x^2의 미분 결과는 2x
    return 2 * x

x = 3
print('수학적 미분 값 : ', derivative_f(x))

수학적 미분 값 :  6


## 3. 수치미분

수치미분은 공식을 직접 전개하지 않고, 함수값의 변화를 이용해 미분값을 근사하는 방법이다.

즉, x를 아주 조금 움직였을 때 함수값이 얼마나 변하는지를 이용해 기울기를 추정하는 방식이다.

1. 전방차분
현재 점 x와 바로 앞이 아니라, x보다 조금 큰 점 x+h를 이용한다.

$$
f'(x) \approx \frac{f(x+h) - f(x)}{h}
$$

2. 후방차분
현재 점 x와 x보다 조금 작은 점 x-h를 이용한다.

$$
f'(x) \approx \frac{f(x) - f(x-h)}{h}
$$

3. 중앙차분
현재 점 x의 양쪽인 x-h, x+h를 함께 이용한다.

$$
f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}
$$

중앙차분은 양쪽 정보를 함께 사용하므로 보통 전방차분, 후방차분보다 더 정확한 근사를 제공한다.

In [4]:
# 전방 차분
def forward_diff(func, x, h=1e-4):
    return (func(x + h) - func(x)) / h

# 후방 차분
def backward_diff(func, x, h=1e-4):
    return (func(x) - func(x - h)) / h

# 중앙 차분
def central_diff(func, x, h=1e-4):
    return (func(x + h) - func(x - h)) / (2 * h)

x = 3

print('전방차분 : ', forward_diff(f, x))
print('후방차분 : ', backward_diff(f, x))
print('중앙차분 : ', central_diff(f, x))
print('수학적 미분값:', derivative_f(x))

전방차분 :  6.000100000012054
후방차분 :  5.99990000001327
중앙차분 :  6.000000000012662
수학적 미분값: 6


## 4. 기울기(gradient)란

딥러닝 모델에서는 가중치, 편향처럼 조정해야 할 값이 여러 개 있다.

이때 각 변수에 대해 손실이 얼마나 민감하게 변하는지를 모아놓은 것이 기울기(gradient)이다.

예를 들어 손실 $L(w, b)$가 있을 때

- $\frac{\partial L}{\partial w}$ : 가중치 $w$를 조금 바꿨을 때 손실이 얼마나 변하는가
- $\frac{\partial L}{\partial b}$ : 편향 $b$를 조금 바꿨을 때 손실이 얼마나 변하는가

를 의미한다.

즉 gradient는 “어느 방향으로 움직이면 손실이 줄어들지” 알려주는 정보라고 이해하면 된다.

## 5. 간단한 손실 함수 예시

가중치 $w$ 하나를 가진 아주 단순한 모델을 생각해보자.

예측값을

$$
\hat{y} = wx
$$

라고 하고,
정답과의 차이를 제곱오차로 나타내면

$$
L = (\hat{y} - y)^2
$$

가 된다.

이 경우 손실 $L$은
가중치 $w$에 따라 달라진다.

즉 손실을 줄이려면
$w$를 어느 방향으로 바꿔야 하는지 알아야 하고,
그때 필요한 값이 바로 $\frac{dL}{dw}$이다.


In [5]:
x = 2
y = 8
w = 3

y_hat = w * x               # 예측값
loss = (y_hat - y) ** 2     # 손실값

print('예측값: ', y_hat)
print('손실값: ', loss)


예측값:  6
손실값:  4


In [6]:
def simple_loss(w):
    x = 2
    y = 8
    y_hat = w * x
    return (y_hat - y) ** 2

for w in [2.0, 3.0, 4.0, 5.0]:
    print(f'{w=}, loss={simple_loss(w)}')

w=2.0, loss=16.0
w=3.0, loss=4.0
w=4.0, loss=0.0
w=5.0, loss=4.0


## 6. PyTorch의 자동미분(autograd)

PyTorch에서는 `requires_grad=True`를 설정하면
해당 tensor에 대해 gradient를 계산할 수 있다.

즉 우리가 미분 공식을 매번 손으로 전개하지 않아도,
PyTorch가 연산 그래프를 따라 gradient를 계산해준다.

이 기능을 자동미분(autograd)이라고 한다.

In [ ]:
# 미분 대상 tensor
x = torch.tensor(4.0, requires_grad=True)

y = x ** 2

print('y:', y)
print('x.grad (backward 전) :', x.grad)

# backward()를 호출하면 y를 x에 대해 미분한 값이 x.grad에 저장 된다.
y.backward()
print('x.grad (backward 후) :', x.grad)


y: tensor(16., grad_fn=<PowBackward0>)
x.grad (backward 전) : None
x.grad (backward 후) : tensor(8.)


## 7. 편미분과 여러 변수의 gradient

딥러닝에서는 보통 변수 하나가 아니라 여러 파라미터가 함께 존재한다.

예를 들어

$$
z = wx + b
$$

처럼 가중치 $w$와 편향 $b$가 함께 있을 때는,
각 변수에 대해 따로 gradient를 구하게 된다.


In [14]:
# w와 b 두 변수에 대한 gradient 확인
x = torch.tensor(2.0)
y_true = torch.tensor(7.0)

w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

# 선형식 z = wx + b
y_pred = w * x + b

# 손실 = (예측 - 정답)^2
loss = (y_pred - y_true) ** 2

print('y_pred:', y_pred)
print('loss:', loss)

loss.backward()

print('dL/dw:', w.grad)
print('dL/db:', b.grad)

# loss = (wx + b - y_true)^2
# b에 대한 미분 => 출력에 1배 영향
# dL/db = 2(wx + b - y_true) * 1
# w에 대한 미분 => 출력에 x배 영향
# dL/dw = 2(wx + b - y_true) * x

y_pred: tensor(7., grad_fn=<AddBackward0>)
loss: tensor(0., grad_fn=<PowBackward0>)
dL/dw: tensor(0.)
dL/db: tensor(0.)


## 8. gradient의 부호의 의미

gradient 값의 부호는 손실을 줄이기 위해 파라미터를 어느 방향으로 움직여야 하는지에 대한 힌트를 준다.

- gradient가 양수이면: 파라미터를 줄이는 방향이 손실 감소에 유리할 수 있다.
- gradient가 음수이면: 파라미터를 늘리는 방향이 손실 감소에 유리할 수 있다.

즉 gradient는 단순한 숫자가 아니라 손실을 줄이기 위한 방향 정보라고 볼 수 있다.

In [15]:
# 서로 다른 w 값에서 손실과 gradient가 어떻게 달라지는지 확인
for intial_w in [1.0, 2.5, 3.5, 5.0]:
    w = torch.tensor(intial_w, requires_grad=True)
    b = torch.tensor(1.0, requires_grad=True)

    y_pred = w * x + b
    loss = (y_pred - y_true) ** 2
    loss.backward()

    print(f'w={intial_w}, loss={loss.item()}, dL/dW={w.grad.item()}')

w=1.0, loss=16.0, dL/dW=-16.0
w=2.5, loss=1.0, dL/dW=-4.0
w=3.5, loss=1.0, dL/dW=4.0
w=5.0, loss=16.0, dL/dW=16.0


## 정리

1. 미분은 입력이 조금 변할 때 출력이 얼마나 변하는지 보는 도구이다.
2. gradient는 여러 파라미터에 대한 변화율을 모아놓은 값이다.
3. 손실을 줄이려면 파라미터를 어느 방향으로 바꿔야 하는지 알아야 한다.
4. PyTorch는 autograd를 통해 gradient를 자동으로 계산할 수 있다.
5. 모델 학습에서는 이 gradient를 바탕으로 파라미터를 업데이트하게 된다.